# Week 2 practical: real Week 1 HTTP backend

This follows `01-gateway-contracts-and-fake-backend.ipynb`. The fake proves gateway logic quickly; this notebook proves the actual adapter can call the Qwen-backed Week 1 server.

Before running: start the Week 1 server in another terminal:

```bash
cd /Users/myatkaung/Desktop/production-llm-inference-lab
source .venv/bin/activate
uvicorn server:app --app-dir week-01-baseline-server --host 127.0.0.1 --port 8001
```

Source alignment: HLSO Chapter 3 and the Class 2 engine-to-gateway learning pattern; see `../NOTEBOOK-SOURCES.md`.

In [1]:
import json
import uuid
from dataclasses import dataclass
from typing import AsyncIterator, Literal

import httpx
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

BACKEND_URL = 'http://127.0.0.1:8001'
MAX_OUTPUT_TOKENS = 64


/Users/myatkaung/Desktop/production-llm-inference-lab/.venv/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 1. Confirm this is a real backend

This check is deliberately outside the gateway. If it fails, start the Week 1 server first; do not debug the adapter yet.

In [2]:
health = httpx.get(f'{BACKEND_URL}/health', timeout=5)
health.raise_for_status()
print(health.json())
assert health.json()['status'] == 'ok'


{'status': 'ok', 'model_loaded': False, 'model_id': 'Qwen/Qwen2.5-0.5B-Instruct', 'device': None}


## 2. Define public and internal contracts

The public client uses `max_tokens`; the Week 1 backend uses `max_new_tokens`. The adapter owns that translation.

In [3]:
class ChatMessage(BaseModel):
    role: Literal['system', 'user', 'assistant']
    content: str = Field(min_length=1)

class ChatCompletionRequest(BaseModel):
    model: str = Field(min_length=1)
    messages: list[ChatMessage] = Field(min_length=1)
    max_tokens: int = Field(default=32, ge=1)
    stream: bool = False

class BackendGenerateRequest(BaseModel):
    messages: list[ChatMessage]
    max_new_tokens: int

@dataclass
class BackendResult:
    text: str
    prompt_tokens: int
    output_tokens: int
    backend_model_id: str

def to_backend_request(request: ChatCompletionRequest):
    return BackendGenerateRequest(messages=request.messages, max_new_tokens=request.max_tokens)


## 3. The real HTTP adapter

Unlike `FakeBackend`, this adapter serializes the internal request, crosses the HTTP boundary, checks the response, and parses the Week 1 JSON/SSE contracts.

In [4]:
class HttpWeek1Backend:
    def __init__(self, base_url=BACKEND_URL):
        self.base_url = base_url

    async def generate(self, request: BackendGenerateRequest) -> BackendResult:
        async with httpx.AsyncClient(base_url=self.base_url, timeout=45) as client:
            response = await client.post('/generate', json=request.model_dump())
        response.raise_for_status()
        body = response.json()
        return BackendResult(body['text'], body['prompt_tokens'], body['output_tokens'], body['model_id'])

    async def stream(self, request: BackendGenerateRequest) -> AsyncIterator[str]:
        async with httpx.AsyncClient(base_url=self.base_url, timeout=45) as client:
            async with client.stream('POST', '/generate/stream', json=request.model_dump()) as response:
                response.raise_for_status()
                async for line in response.aiter_lines():
                    if not line.startswith('data: '):
                        continue
                    payload = line.removeprefix('data: ')
                    if payload == '[DONE]':
                        return
                    yield json.loads(payload)['text']


## 4. Gateway application: the client still calls one public endpoint


In [5]:
def build_gateway(backend):
    app = FastAPI()

    @app.post('/v1/chat/completions')
    async def chat_completions(request: ChatCompletionRequest):
        request_id = f'req_{uuid.uuid4().hex[:12]}'
        if request.max_tokens > MAX_OUTPUT_TOKENS:
            raise HTTPException(400, detail={'type': 'output_token_limit', 'request_id': request_id})
        internal = to_backend_request(request)
        if not request.stream:
            result = await backend.generate(internal)
            return {'id': request_id, 'object': 'chat.completion', 'model': request.model, 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': result.text}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': result.prompt_tokens, 'completion_tokens': result.output_tokens, 'total_tokens': result.prompt_tokens + result.output_tokens}}

        async def events():
            async for text in backend.stream(internal):
                yield 'data: ' + json.dumps({'id': request_id, 'choices': [{'index': 0, 'delta': {'content': text}}]}) + '\n\n'
            yield 'data: [DONE]\n\n'
        return StreamingResponse(events(), media_type='text/event-stream')

    return app


## 5. Real non-streaming integration

This request enters the gateway, crosses HTTP to the Qwen server on port 8001, returns through the adapter, and is mapped to the public response.


In [6]:
client = TestClient(build_gateway(HttpWeek1Backend()))
response = client.post('/v1/chat/completions', json={'model': 'local-qwen', 'messages': [{'role': 'user', 'content': 'Explain caching in one sentence.'}], 'max_tokens': 16})
response.raise_for_status()
body = response.json()
assert body['id'].startswith('req_')
assert body['choices'][0]['message']['content']
assert body['usage']['prompt_tokens'] > 0
assert body['usage']['completion_tokens'] > 0
print(body)


{'id': 'req_a58436a2f157', 'object': 'chat.completion', 'model': 'local-qwen', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': 'Caching is the process of storing frequently accessed data in memory to reduce the number'}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 36, 'completion_tokens': 16, 'total_tokens': 52}}


## 6. Real streaming integration

The exact same public endpoint now uses `stream: true`. The adapter reads real backend SSE events and the gateway emits its own public SSE deltas in order.


In [7]:
stream_response = client.post('/v1/chat/completions', json={'model': 'local-qwen', 'messages': [{'role': 'user', 'content': 'Explain caching in one sentence.'}], 'max_tokens': 8, 'stream': True})
stream_response.raise_for_status()
events = [line.removeprefix('data: ') for line in stream_response.text.splitlines() if line.startswith('data: ')]
payloads = [json.loads(event) for event in events[:-1]]
text = ''.join(payload['choices'][0]['delta']['content'] for payload in payloads)
assert events[-1] == '[DONE]'
assert text
print({'chunks': len(payloads), 'text': text})


{'chunks': 8, 'text': 'Caching is the process of storing frequently'}


## 7. What this proves and what it does not

**Proves:** the public gateway contract translates to the real Week 1 HTTP JSON/SSE contract, preserves a non-empty answer, and streams an ordered completion.

**Does not yet prove:** production timeout cancellation, queue time, concurrency behavior, Prometheus metrics, or resilience after a backend crash. Those are the next Week 2 exercises.

**Your task:** stop the Week 1 server, rerun the non-streaming cell, and record the error. Then restart it and explain why this is an integration finding, not a fake-backend contract failure.